# Phase 4: SVM with Data Augmentation Comparison
**CSC14120 - Parallel Programming**

## Objective
Compare SVM classification accuracy when training with:
1. **Original features** - No augmentation
2. **Augmented features** - Multiple augmented views per image

## Augmentation Techniques
- **Random Horizontal Flip** - Mirror images left-right
- **Random Crop** - Crop with padding=4, then resize to 32x32
- **Cutout** - Random rectangular mask
- **Color Jitter** - Brightness/contrast variations

## Hypothesis
Augmented training data provides more diverse viewpoints, potentially improving SVM generalization.

## 1. Setup

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import os
import glob
import shutil
import numpy as np
import struct
import time
import random
import gc

# Find and copy project
input_dir = '/kaggle/input'
src_dirs = glob.glob(f'{input_dir}/**/src', recursive=True)

if src_dirs:
    project_dir = os.path.dirname(src_dirs[0])
    print(f"Found project at: {project_dir}")
    shutil.copytree(project_dir, '/kaggle/working/project', dirs_exist_ok=True)
    
    for root, dirs, _ in os.walk('/kaggle/working/project'):
        if 'src' in dirs:
            os.chdir(root)
            break
else:
    print("ERROR: Project not found")

print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Download CIFAR-10
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/

print('CIFAR-10 ready!')

In [ ]:
# Find weights file
weights_file = 'phase3_opt.weights'

if not os.path.exists(weights_file):
    weight_files = glob.glob('/kaggle/input/**/*.weights', recursive=True)
    if weight_files:
        shutil.copy(weight_files[0], weights_file)
        print(f"Copied weights from: {weight_files[0]}")
    else:
        print("ERROR: No weights file found!")
else:
    print(f"Found weights: {weights_file}")

!ls -lh {weights_file}

## 2. Build Feature Extractor

In [ ]:
# Build LIBSVM
print("Building LIBSVM...")
!git clone --depth 1 https://github.com/cjlin1/libsvm.git libsvm_src 2>/dev/null || echo "Already cloned"
!cd libsvm_src && make lib
!cp libsvm_src/svm.h include/ 2>/dev/null || true
!cp libsvm_src/svm.cpp src/ 2>/dev/null || true

print("\nBuilding feature_extractor...")
!nvcc -O3 -std=c++17 -arch=sm_70 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DWITH_SVM -DWITH_LIBSVM \
    -Iinclude -Ilibsvm_src -lcublas -lcudnn \
    -o feature_extractor \
    src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu \
    src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp libsvm_src/svm.cpp

!if [ -f feature_extractor ]; then echo "Build SUCCESS!"; else echo "Build FAILED!"; fi

In [ ]:
# Extract features (original, no augmentation)
print("Extracting features with GPU autoencoder...")
start = time.time()
!./feature_extractor --data data --weights {weights_file} --extract-only
print(f"\nFeature extraction: {time.time() - start:.2f}s")
!ls -lh *.bin 2>/dev/null || echo "No .bin files"

## 3. Load Data

In [ ]:
# Load features and labels
feature_dim = 8192

print("Loading features...")
train_features_orig = np.fromfile('train_features.bin', dtype=np.float32).reshape(-1, feature_dim)
test_features = np.fromfile('test_features.bin', dtype=np.float32).reshape(-1, feature_dim)

print(f"Train features: {train_features_orig.shape}")
print(f"Test features: {test_features.shape}")

# Load labels
def load_cifar10_labels(data_dir):
    train_labels, test_labels = [], []
    for i in range(1, 6):
        with open(f"{data_dir}/data_batch_{i}.bin", 'rb') as f:
            for _ in range(10000):
                train_labels.append(struct.unpack('B', f.read(1))[0])
                f.read(3072)
    with open(f"{data_dir}/test_batch.bin", 'rb') as f:
        for _ in range(10000):
            test_labels.append(struct.unpack('B', f.read(1))[0])
            f.read(3072)
    return np.array(train_labels), np.array(test_labels)

train_labels_orig, test_labels = load_cifar10_labels('data')
print(f"Labels loaded: {len(train_labels_orig)} train, {len(test_labels)} test")

In [ ]:
# Load raw images for augmentation
def load_cifar10_images(data_dir):
    """Load raw CIFAR-10 images as numpy arrays (HWC format)"""
    train_images = []
    
    for i in range(1, 6):
        with open(f"{data_dir}/data_batch_{i}.bin", 'rb') as f:
            for _ in range(10000):
                f.read(1)  # skip label
                img = np.frombuffer(f.read(3072), dtype=np.uint8)
                img = img.reshape(3, 32, 32).transpose(1, 2, 0)  # CHW -> HWC
                train_images.append(img)
    
    return np.array(train_images)

train_images_raw = load_cifar10_images('data')
print(f"Raw images loaded: {train_images_raw.shape}")

## 4. Data Augmentation Functions

In [ ]:
def random_horizontal_flip(img, p=0.5):
    """Randomly flip image horizontally"""
    if random.random() < p:
        return np.fliplr(img).copy()
    return img.copy()

def random_crop(img, padding=4):
    """Random crop with padding"""
    h, w = img.shape[:2]
    padded = np.pad(img, ((padding, padding), (padding, padding), (0, 0)), mode='reflect')
    top = random.randint(0, 2 * padding)
    left = random.randint(0, 2 * padding)
    return padded[top:top+h, left:left+w]

def cutout(img, size=8):
    """Random rectangular cutout"""
    h, w = img.shape[:2]
    img = img.copy()
    cx, cy = random.randint(0, w), random.randint(0, h)
    x1, x2 = max(0, cx - size//2), min(w, cx + size//2)
    y1, y2 = max(0, cy - size//2), min(h, cy + size//2)
    img[y1:y2, x1:x2] = 128
    return img

def color_jitter(img, brightness=0.2, contrast=0.2):
    """Random brightness and contrast"""
    img = img.astype(np.float32)
    b = 1.0 + random.uniform(-brightness, brightness)
    c = 1.0 + random.uniform(-contrast, contrast)
    img = img * b
    mean = img.mean()
    img = (img - mean) * c + mean
    return np.clip(img, 0, 255).astype(np.uint8)

def augment_image(img, use_flip=True, use_crop=True, use_cutout=False, use_color=False):
    """Apply augmentation pipeline"""
    if use_flip:
        img = random_horizontal_flip(img)
    if use_crop:
        img = random_crop(img, padding=4)
    if use_cutout:
        img = cutout(img, size=8)
    if use_color:
        img = color_jitter(img)
    return img

# Visualize augmentation
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
sample_img = train_images_raw[0]

axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i in range(1, 5):
    aug_img = augment_image(sample_img, use_flip=True, use_crop=True)
    axes[0, i].imshow(aug_img)
    axes[0, i].set_title(f'Flip+Crop {i}')
    axes[0, i].axis('off')

for i in range(5):
    aug_img = augment_image(sample_img, use_flip=True, use_crop=True, use_cutout=True, use_color=True)
    axes[1, i].imshow(aug_img)
    axes[1, i].set_title(f'Full Aug {i}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('augmentation_examples.png', dpi=150)
plt.show()

## 5. Feature-Space Augmentation

Since re-extracting features for augmented images requires GPU encoder passes, we use **feature-space augmentation** as an efficient alternative:

1. **Noise injection** - Add small Gaussian noise to features
2. **Feature dropout** - Randomly zero out some feature dimensions
3. **Mixup** - Interpolate between features of same class

This simulates the effect of image augmentation at the feature level.

In [ ]:
def feature_noise(features, noise_std=0.1):
    """Add Gaussian noise to features"""
    noise = np.random.randn(*features.shape).astype(np.float32) * noise_std
    return features + noise

def feature_dropout(features, drop_rate=0.1):
    """Randomly zero out feature dimensions"""
    mask = np.random.random(features.shape) > drop_rate
    return features * mask / (1 - drop_rate)  # Scale to maintain magnitude

def feature_mixup(features, labels, alpha=0.2):
    """Mixup: interpolate between samples of same class"""
    n = len(features)
    augmented = []
    aug_labels = []
    
    for i in range(n):
        # Find another sample of same class
        same_class = np.where(labels == labels[i])[0]
        j = np.random.choice(same_class)
        
        # Mixup
        lam = np.random.beta(alpha, alpha)
        mixed = lam * features[i] + (1 - lam) * features[j]
        augmented.append(mixed)
        aug_labels.append(labels[i])
    
    return np.array(augmented), np.array(aug_labels)

def augment_features(features, labels, n_augments=2, noise_std=0.05, drop_rate=0.05):
    """Generate augmented features"""
    all_features = [features]
    all_labels = [labels]
    
    for _ in range(n_augments):
        # Apply noise
        aug_feat = feature_noise(features.copy(), noise_std)
        # Apply dropout
        aug_feat = feature_dropout(aug_feat, drop_rate)
        all_features.append(aug_feat)
        all_labels.append(labels.copy())
    
    return np.vstack(all_features), np.concatenate(all_labels)

print("Feature augmentation functions defined.")

## 6. Preprocessing (PCA)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

N_COMPONENTS = 640

print("="*60)
print("PREPROCESSING")
print("="*60)

# Standardize
print("\n1. Standardizing features...")
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features_orig)
test_scaled = scaler.transform(test_features)

# PCA
print(f"2. Applying PCA: {feature_dim} -> {N_COMPONENTS} dimensions...")
pca = PCA(n_components=N_COMPONENTS, random_state=42)
train_pca = pca.fit_transform(train_scaled)
test_pca = pca.transform(test_scaled)

print(f"   Explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"   Train PCA shape: {train_pca.shape}")
print(f"   Test PCA shape: {test_pca.shape}")

# Free memory
del train_features_orig, train_scaled, test_scaled
gc.collect()

## 7. Build ThunderSVM

In [ ]:
import sys

print("Building ThunderSVM from source...")
!rm -rf thundersvm
!git clone --depth 1 https://github.com/Xtra-Computing/thundersvm.git
!cd thundersvm && mkdir -p build && cd build && cmake .. -DUSE_CUDA=ON -DCMAKE_BUILD_TYPE=Release && make -j4
!cd thundersvm/python && pip install -e .

USE_THUNDER = False
try:
    sys.path.insert(0, 'thundersvm/python')
    from thundersvm import SVC as ThunderSVC
    print("\nThunderSVM installed successfully!")
    USE_THUNDER = True
except ImportError as e:
    print(f"ThunderSVM failed: {e}")
    print("Using sklearn SVC")
    USE_THUNDER = False

In [ ]:
from sklearn.svm import SVC as SklearnSVC
from sklearn.metrics import accuracy_score, classification_report

if USE_THUNDER:
    from thundersvm import SVC as ThunderSVC
    SVC = ThunderSVC
    print("Using ThunderSVM")
else:
    SVC = SklearnSVC
    print("Using sklearn SVC")

## 8. Experiment 1: SVM WITHOUT Augmentation (Baseline)

In [ ]:
# Sample training data (stratified)
TRAIN_SAMPLES = 50000
samples_per_class = TRAIN_SAMPLES // 10

print(f"Stratified sampling: {samples_per_class} per class = {TRAIN_SAMPLES} total")

np.random.seed(42)
train_indices = []
for c in range(10):
    class_idx = np.where(train_labels_orig == c)[0]
    selected = np.random.choice(class_idx, size=samples_per_class, replace=False)
    train_indices.extend(selected)

train_indices = np.array(train_indices)
np.random.shuffle(train_indices)

X_train_base = train_pca[train_indices]
y_train_base = train_labels_orig[train_indices]

print(f"Baseline training set: {X_train_base.shape}")

In [ ]:
print("="*60)
print("EXPERIMENT 1: SVM WITHOUT AUGMENTATION (BASELINE)")
print("="*60)

# SVM parameters
KERNEL = 'rbf'
C = 10
GAMMA = 'scale'

if USE_THUNDER:
    gamma_val = 1.0 / (X_train_base.shape[1] * X_train_base.var())
    svm_base = SVC(kernel=KERNEL, C=C, gamma=gamma_val, verbose=True)
else:
    svm_base = SVC(kernel=KERNEL, C=C, gamma=GAMMA, cache_size=2000, verbose=True)

print(f"\nTraining SVM on {len(X_train_base)} samples...")
start = time.time()
svm_base.fit(X_train_base, y_train_base)
train_time_base = time.time() - start
print(f"Training time: {train_time_base:.2f}s")

# Evaluate
print("\nEvaluating...")
start = time.time()
pred_base = svm_base.predict(test_pca)
eval_time_base = time.time() - start

acc_base = accuracy_score(test_labels, pred_base)
print(f"\n*** BASELINE ACCURACY: {acc_base*100:.2f}% ***")
print(f"Eval time: {eval_time_base:.2f}s")

## 9. Experiment 2: SVM WITH Feature Augmentation

In [ ]:
print("="*60)
print("EXPERIMENT 2: SVM WITH FEATURE AUGMENTATION")
print("="*60)

# Generate augmented features
N_AUGMENTS = 2  # 2 augmented copies + 1 original = 3x data
NOISE_STD = 0.05
DROP_RATE = 0.05

print(f"\nGenerating {N_AUGMENTS} augmented copies per sample...")
print(f"  Noise std: {NOISE_STD}")
print(f"  Dropout rate: {DROP_RATE}")

X_train_aug, y_train_aug = augment_features(
    X_train_base, y_train_base, 
    n_augments=N_AUGMENTS, 
    noise_std=NOISE_STD, 
    drop_rate=DROP_RATE
)

print(f"Augmented training set: {X_train_aug.shape}")
print(f"  Original: {len(X_train_base)} -> Augmented: {len(X_train_aug)}")

In [ ]:
# Train SVM with augmented data
if USE_THUNDER:
    gamma_val = 1.0 / (X_train_aug.shape[1] * X_train_aug.var())
    svm_aug = SVC(kernel=KERNEL, C=C, gamma=gamma_val, verbose=True)
else:
    svm_aug = SVC(kernel=KERNEL, C=C, gamma=GAMMA, cache_size=2000, verbose=True)

print(f"\nTraining SVM on {len(X_train_aug)} augmented samples...")
start = time.time()
svm_aug.fit(X_train_aug, y_train_aug)
train_time_aug = time.time() - start
print(f"Training time: {train_time_aug:.2f}s")

# Evaluate
print("\nEvaluating...")
start = time.time()
pred_aug = svm_aug.predict(test_pca)
eval_time_aug = time.time() - start

acc_aug = accuracy_score(test_labels, pred_aug)
print(f"\n*** AUGMENTED ACCURACY: {acc_aug*100:.2f}% ***")
print(f"Eval time: {eval_time_aug:.2f}s")

## 10. Experiment 3: SVM with Mixup Augmentation

In [ ]:
print("="*60)
print("EXPERIMENT 3: SVM WITH MIXUP AUGMENTATION")
print("="*60)

# Generate mixup features
print("\nGenerating mixup augmented features...")
X_mixup, y_mixup = feature_mixup(X_train_base, y_train_base, alpha=0.2)

# Combine original + mixup
X_train_mixup = np.vstack([X_train_base, X_mixup])
y_train_mixup = np.concatenate([y_train_base, y_mixup])

print(f"Mixup training set: {X_train_mixup.shape}")

In [ ]:
# Train SVM with mixup data
if USE_THUNDER:
    gamma_val = 1.0 / (X_train_mixup.shape[1] * X_train_mixup.var())
    svm_mixup = SVC(kernel=KERNEL, C=C, gamma=gamma_val, verbose=True)
else:
    svm_mixup = SVC(kernel=KERNEL, C=C, gamma=GAMMA, cache_size=2000, verbose=True)

print(f"\nTraining SVM on {len(X_train_mixup)} mixup samples...")
start = time.time()
svm_mixup.fit(X_train_mixup, y_train_mixup)
train_time_mixup = time.time() - start
print(f"Training time: {train_time_mixup:.2f}s")

# Evaluate
print("\nEvaluating...")
start = time.time()
pred_mixup = svm_mixup.predict(test_pca)
eval_time_mixup = time.time() - start

acc_mixup = accuracy_score(test_labels, pred_mixup)
print(f"\n*** MIXUP ACCURACY: {acc_mixup*100:.2f}% ***")
print(f"Eval time: {eval_time_mixup:.2f}s")

## 11. Results Comparison

In [ ]:
print("\n" + "="*70)
print("RESULTS COMPARISON: SVM WITH vs WITHOUT AUGMENTATION")
print("="*70)

results = {
    'Method': ['Baseline (No Aug)', 'Feature Aug (Noise+Dropout)', 'Mixup Aug'],
    'Train Samples': [len(X_train_base), len(X_train_aug), len(X_train_mixup)],
    'Accuracy (%)': [acc_base*100, acc_aug*100, acc_mixup*100],
    'Train Time (s)': [train_time_base, train_time_aug, train_time_mixup],
    'Eval Time (s)': [eval_time_base, eval_time_aug, eval_time_mixup]
}

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print("\n" + "-"*70)
print("ANALYSIS:")
print("-"*70)

best_method = df_results.loc[df_results['Accuracy (%)'].idxmax(), 'Method']
best_acc = df_results['Accuracy (%)'].max()

print(f"Best method: {best_method} with {best_acc:.2f}% accuracy")
print(f"")
print(f"Baseline vs Feature Aug: {acc_aug*100 - acc_base*100:+.2f}%")
print(f"Baseline vs Mixup Aug:   {acc_mixup*100 - acc_base*100:+.2f}%")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

methods = ['Baseline', 'Feature Aug', 'Mixup']
accuracies = [acc_base*100, acc_aug*100, acc_mixup*100]
train_times = [train_time_base, train_time_aug, train_time_mixup]
colors = ['#3498db', '#2ecc71', '#e74c3c']

# Accuracy comparison
bars1 = axes[0].bar(methods, accuracies, color=colors, edgecolor='black')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Test Accuracy Comparison')
axes[0].set_ylim([min(accuracies)-5, max(accuracies)+5])
for bar, acc in zip(bars1, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')

# Training time comparison
bars2 = axes[1].bar(methods, train_times, color=colors, edgecolor='black')
axes[1].set_ylabel('Training Time (s)')
axes[1].set_title('Training Time Comparison')
for bar, t in zip(bars2, train_times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 f'{t:.1f}s', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('augmentation_comparison.png', dpi=150)
plt.show()

In [ ]:
# Per-class comparison
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("\nPer-class Accuracy Comparison:")
print("-"*60)
print(f"{'Class':<12} {'Baseline':>10} {'Feature Aug':>12} {'Mixup':>10}")
print("-"*60)

for i, name in enumerate(class_names):
    mask = test_labels == i
    acc_b = np.mean(pred_base[mask] == test_labels[mask]) * 100
    acc_a = np.mean(pred_aug[mask] == test_labels[mask]) * 100
    acc_m = np.mean(pred_mixup[mask] == test_labels[mask]) * 100
    print(f"{name:<12} {acc_b:>10.2f}% {acc_a:>12.2f}% {acc_m:>10.2f}%")

print("-"*60)
print(f"{'OVERALL':<12} {acc_base*100:>10.2f}% {acc_aug*100:>12.2f}% {acc_mixup*100:>10.2f}%")

## 12. Conclusion

In [ ]:
print("="*70)
print("CONCLUSION")
print("="*70)
print("""
This experiment compared SVM classification with and without data augmentation
using features extracted from a trained autoencoder.

KEY FINDINGS:
""")

if acc_aug > acc_base:
    print(f"1. Feature augmentation IMPROVED accuracy by {(acc_aug-acc_base)*100:.2f}%")
else:
    print(f"1. Feature augmentation did NOT improve accuracy ({(acc_aug-acc_base)*100:.2f}%)")

if acc_mixup > acc_base:
    print(f"2. Mixup augmentation IMPROVED accuracy by {(acc_mixup-acc_base)*100:.2f}%")
else:
    print(f"2. Mixup augmentation did NOT improve accuracy ({(acc_mixup-acc_base)*100:.2f}%)")

print(f"""
3. Training time increased proportionally with augmented data size
4. Best performing method: {best_method} ({best_acc:.2f}%)

INTERPRETATION:
- Feature-space augmentation simulates diverse viewpoints without re-encoding
- Results depend on the quality of learned features from autoencoder
- For better results, consider image-space augmentation before feature extraction
""")

# Save results
df_results.to_csv('augmentation_comparison_results.csv', index=False)
shutil.copy('augmentation_comparison_results.csv', '/kaggle/working/')
shutil.copy('augmentation_comparison.png', '/kaggle/working/')
shutil.copy('augmentation_examples.png', '/kaggle/working/')
print("\nResults saved to /kaggle/working/")